In [47]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

print("Iniciando flujo de Ingeniería Analítica...")

# =====================================================================
# ARQUITECTURA DE RUTAS RELATIVAS (Ref: Diapositivas 7 y 11)
# Objetivo: Garantizar que el código corra en cualquier máquina.
# =====================================================================

# FORMA INCORRECTA (Ruta Absoluta y dependiente del usuario)
# ruta_base = "C:/Users/Andres/Desktop/Clase3/datos/"
# df = pd.read_csv(ruta_base + "archivo.csv")

# FORMA CORRECTA (Ruta Relativa dinámica con pathlib)
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Crear la estructura de directorios automáticamente si no existe
for directorio in [RAW_DIR, PROCESSED_DIR]:
    directorio.mkdir(parents=True, exist_ok=True)

print("1. Rutas relativas validadas y directorios creados.")

Iniciando flujo de Ingeniería Analítica...
1. Rutas relativas validadas y directorios creados.


In [48]:
# =====================================================================
# LECTURA ESTRUCTURADA: Interceptando el Caos (Ref: Diapositiva 6)
# Objetivo: Cargar datos sin perder información vital como los ceros.
# =====================================================================

# FORMA INCORRECTA (Pérdida de ceros iniciales en códigos DPA)
df_catalogo_malo = pd.read_csv(RAW_DIR / "catalogo_municipal.csv")

# FORMA CORRECTA (Definición estricta de tipos y separadores)
df_catalogo = pd.read_csv(
    RAW_DIR / "catalogo_municipal.csv",
    sep=";",
    dtype={"codigo_dpa": str, "id_cliente": str} # Protege ceros a la izquierda
)
print(f"2. Lectura estructurada exitosa: {len(df_catalogo)} registros en catálogo.")
print(df_catalogo_malo)
print(df_catalogo)

2. Lectura estructurada exitosa: 5 registros en catálogo.
  id_cliente;codigo_dpa;region;estado_registro
0                       C001;0101;Norte;Activo
1                         C002;0701;Sur;Activo
2                       C003;1101;Sur;Inactivo
3                      C004;1701;Centro;Activo
4                       C005;0901;Costa;Activo
  id_cliente codigo_dpa  region estado_registro
0       C001       0101   Norte          Activo
1       C002       0701     Sur          Activo
2       C003       1101     Sur        Inactivo
3       C004       1701  Centro          Activo
4       C005       0901   Costa          Activo


In [49]:
# =====================================================================
# MATRIZ DE DECISIÓN: CONCAT vs MERGE (Ref: Diapositiva 4)
# Contexto: Unir ventas de enero y febrero (Integración Estructural)
# =====================================================================

# FORMA INCORRECTA (Usar Merge para apilar datos con la misma estructura)
df_enero = pd.read_csv(RAW_DIR / "ventas_enero.csv")
df_febrero = pd.read_csv(RAW_DIR / "ventas_febrero.csv")
merge_sin_sentido = pd.merge(df_enero, df_febrero, on="id_transaccion") # Falla lógica
print(df_enero)
print(df_febrero)
print(merge_sin_sentido)

# FORMA CORRECTA (Uso de CONCAT para apilar e inyectar trazabilidad)
df_enero = pd.read_csv(RAW_DIR / "ventas_enero.csv", dtype={"id_cliente": str})
df_febrero = pd.read_csv(RAW_DIR / "ventas_febrero.csv", dtype={"id_cliente": str})

# Añadimos metadatos para saber de dónde vino cada fila tras unirlas
df_enero["mes_origen"] = "Enero"
df_febrero["mes_origen"] = "Febrero"

df_ventas = pd.concat([df_enero, df_febrero], ignore_index=True)
print(f"3. Concatenación (CONCAT) exitosa: {len(df_ventas)} transacciones totales.")
print(df_ventas)

  id_transaccion id_cliente       fecha  monto
0           T100       C001  2026-01-10  150.5
1           T101       C002  2026-01-15   85.0
2           T102       C001  2026-01-20   40.0
3           T103       C099  2026-01-22  200.0
  id_transaccion id_cliente       fecha   monto
0           T104       C003  2026-02-05  120.00
1           T105       C004  2026-02-14  300.75
2           T106       C002  2026-02-18   65.20
3           T107       C005  2026-02-25   95.00
Empty DataFrame
Columns: [id_transaccion, id_cliente_x, fecha_x, monto_x, id_cliente_y, fecha_y, monto_y]
Index: []
3. Concatenación (CONCAT) exitosa: 8 transacciones totales.
  id_transaccion id_cliente       fecha   monto mes_origen
0           T100       C001  2026-01-10  150.50      Enero
1           T101       C002  2026-01-15   85.00      Enero
2           T102       C001  2026-01-20   40.00      Enero
3           T103       C099  2026-01-22  200.00      Enero
4           T104       C003  2026-02-05  120.00    Feb

In [50]:
# =====================================================================
# ANATOMÍA DE UN MERGE FALLIDO (Error Silencioso) (Ref: Diapositivas 2 y 5)
# Contexto: Cruzar ventas con clientes (Integración Relacional)
# =====================================================================

# FORMA INCORRECTA (Merge a ciegas. Si el catálogo tiene duplicados, las ventas se multiplican)
clientes_sucios = pd.read_csv(RAW_DIR / "clientes_duplicados.csv")
df_explosion = pd.merge(df_ventas, clientes_sucios, on="id_cliente")
print("Falso Total de Ventas:", df_explosion['monto'].sum()) # ¡El total se infla!
print(df_ventas)

# FORMA CORRECTA (Limpieza previa y validación estricta de cardinalidad)
clientes = pd.read_csv(RAW_DIR / "clientes_duplicados.csv", dtype={"id_cliente": str})

# Defensa 1: Eliminar duplicados en la llave maestra del lado derecho
clientes_limpios = clientes.drop_duplicates(subset=["id_cliente"])

# Defensa 2: Parámetros validate e indicator
df_integrado = pd.merge(
    df_ventas,
    clientes_limpios,
    on="id_cliente",
    how="left",
    validate="many_to_one", # Bloquea la ejecución si la tabla derecha tiene duplicados
    indicator=True          # Permite auditar el cruce
)
huerfanos = len(df_integrado[df_integrado["_merge"] == "left_only"])
print(f"4. Merge validado. Clientes huérfanos (sin match en catálogo): {huerfanos}")

# Limpiar la columna técnica de merge para no ensuciar la salida
df_integrado = df_integrado.drop(columns=["_merge"])

Falso Total de Ventas: 1006.65
  id_transaccion id_cliente       fecha   monto mes_origen
0           T100       C001  2026-01-10  150.50      Enero
1           T101       C002  2026-01-15   85.00      Enero
2           T102       C001  2026-01-20   40.00      Enero
3           T103       C099  2026-01-22  200.00      Enero
4           T104       C003  2026-02-05  120.00    Febrero
5           T105       C004  2026-02-14  300.75    Febrero
6           T106       C002  2026-02-18   65.20    Febrero
7           T107       C005  2026-02-25   95.00    Febrero
4. Merge validado. Clientes huérfanos (sin match en catálogo): 1


In [51]:
# =====================================================================
# AGRUPAMIENTO Y AGREGACIÓN: Split-Apply-Combine (Ref: Diapositiva 3)
# =====================================================================

# FORMA INCORRECTA (Agregación simple que pierde el contexto y la varianza)
resumen_pobre = df_integrado.groupby("region")["monto"].mean()
print(resumen_pobre)

# FORMA CORRECTA (Agregación multidimensional robusta)
resumen_robusto = df_integrado.groupby("region").agg(
    total_ventas=("monto", "sum"),
    promedio_ventas=("monto", "mean"),
    desviacion=("monto", "std"),    # Contextualiza el promedio
    transacciones=("monto", "count")
).reset_index()

print("\n5. Resumen Analítico Generado:")
print(resumen_robusto)

region
Centro    300.750000
Costa      95.000000
Norte      95.250000
Sur        90.066667
Name: monto, dtype: float64

5. Resumen Analítico Generado:
   region  total_ventas  promedio_ventas  desviacion  transacciones
0  Centro        300.75       300.750000         NaN              1
1   Costa         95.00        95.000000         NaN              1
2   Norte        190.50        95.250000   78.135299              2
3     Sur        270.20        90.066667   27.749114              3


In [52]:
# =====================================================================
# DISEÑO DE SALIDAS REUTILIZABLES (Ref: Diapositiva 7)
# =====================================================================

# FORMA INCORRECTA (Guardar con índices basura y nombres ambiguos)
df_integrado.to_csv(PROCESSED_DIR / "datos_finales_v3.csv")

# FORMA CORRECTA (Metadatos temporales, nomenclatura clara e index=False)
df_integrado["fecha_procesamiento"] = datetime.now().strftime("%Y-%m-%d")

# Generar un nombre de archivo dinámico
nombre_archivo = PROCESSED_DIR / f"ventas_consolidadas_gold_{datetime.now().strftime('%Y%m%d')}.csv"

df_integrado.to_csv(nombre_archivo, index=False, encoding="utf-8")
print(f"\n6. Flujo documentado. Archivo final exportado en:\n{nombre_archivo}")


6. Flujo documentado. Archivo final exportado en:
/content/data/processed/ventas_consolidadas_gold_20260815.csv


In [53]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   79
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4399.99
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   